# Conformal prediction

Paper-oriented tables and figures for the 90% conformal prediction experiments. The default analysis uses the linear probe, excludes TENX143 and Hibou-L, retains only LEMON ViT-S/8 under the label LEMON, and labels CellBridge all/adapted simply as CellBridge.

Figures 1a–f compare representations separately within each dataset under branch split. Figure 1g compares their equal-weight mean across the six dataset/tasks. Figure 2 fixes CellBridge and compares class split, global graph, and branch split. Figure 3 maps cell-level branch-split cardinality over a held-out HEST spatial region. The displayed data frames provide the corresponding conformal tables for refinement.

In [ ]:
import os
import shutil
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from IPython.display import display
from matplotlib.colors import BoundaryNorm
from matplotlib.lines import Line2D

WORKSPACE = Path(os.environ.get(
    "CELLBRIDGE_WORKSPACE",
    "/lustre/scratch126/cellgen/lotfollahi/so25",
))
RESULTS_ROOT = Path(os.environ.get(
    "CELLBRIDGE_INFERENCE_RESULTS",
    WORKSPACE / "outputs/reproducibility/inference",
))
DATA_ROOT = Path(os.environ.get("CELLBRIDGE_DATA_ROOT", WORKSPACE / "data"))
FIGURE_DIR = Path(os.environ.get(
    "CELLBRIDGE_CONFORMAL_FIGURES",
    WORKSPACE / "outputs/reproducibility/figures/conformal_prediction",
))

# Main switches. Change these and rerun the notebook.
PROBE = "linear"  # "linear" or "knn"
EXCLUDED_HEST_SAMPLES = {"TENX143"}
METHOD_COMPARISON_CP = "branch_split"
SPATIAL_SAMPLE = "TENX116"
SPATIAL_CP = "branch_split"
HISTOLOGY_LEVEL = 4
SPATIAL_WINDOW_PX = 6000
SAVE_FIGURES = True

# Seven paper-facing representations. Add/remove entries here as needed.
METHOD_ORDER = [
    "UNI2-h",
    "Virchow2",
    "VOLTA",
    "LEMON",
    "PAST",
    "SpatialWhisperer",
    "CellBridge",
]
CP_ORDER = ["class_split", "graph", "branch_split"]

# Paper-facing representation palette.
METHOD_COLORS = {
    "UNI2-h": "#CB722F",
    "Virchow2": "#93300C",
    "VOLTA": "#8859AB",
    "LEMON": "#D6A238",
    "PAST": "#E5A3B3",
    "SpatialWhisperer": "#287DC5",
    "CellBridge": "#88C2AB",
}
CP_COLORS = {
    "class_split": "#CB722F",
    "graph": "#8859AB",
    "branch_split": "#88C2AB",
}
CP_LABELS = {
    "class_split": "Class split",
    "graph": "Global graph",
    "branch_split": "Branch split (ours)",
}
DATASET_ORDER = [
    "HEST",
    "Lizard",
    "NuCLS",
    "PanNuke",
    "PhenoCell (all)",
    "PhenoCell (immune)",
]
DATASET_MARKERS = {
    "HEST": "^",
    "Lizard": "s",
    "NuCLS": "D",
    "PanNuke": "o",
    "PhenoCell (all)": "P",
    "PhenoCell (immune)": "X",
}

missing_tex = [name for name in ("latex", "dvipng") if shutil.which(name) is None]
if missing_tex:
    raise RuntimeError(
        "Full LaTeX rendering requires: module load texlive-20220321 "
        + "(missing {})".format(", ".join(missing_tex))
    )

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "figure.dpi": 120,
    "savefig.dpi": 300,
})
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR

## Load and aggregate the conformal metrics

Normalized set size means mean prediction-set cardinality divided by the number of eligible labels for that dataset/task. Thus 0.25 means that the prediction set contains 25% of the available labels on average. HEST is averaged across the retained sections so it contributes one point per representation or conformal type.

In [ ]:
BASELINE_LABELS = {
    "uni": "UNI2-h",
    "virchow2": "Virchow2",
    "lemon": "LEMON",
    "past": "PAST",
    "volta": "VOLTA",
    "spatialwhisperer": "SpatialWhisperer",
}


def paper_method(method, version, arm):
    if method == "cellbridge" or arm.startswith("lora_"):
        size = version.removeprefix("cells_")
        size = size.upper() if size != "all" else "all"
        representation = arm.removeprefix("lora_")
        if size == "all" and representation == "adapted":
            return "CellBridge"
        return "CellBridge {} ({})".format(size, representation)
    return BASELINE_LABELS.get(method, method)


def benchmark_label(dataset, task):
    if dataset == "phenocell":
        return "PhenoCell ({})".format(task)
    return {
        "hest": "HEST",
        "lizard": "Lizard",
        "nucls": "NuCLS",
        "pannuke": "PanNuke",
    }[dataset]


def load_conformal_metrics(root):
    paths = list(root.glob(
        "*/*/*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    paths += list(root.glob(
        "*/*/hest/*/evaluation/*/*/*/conformal_prediction/metrics.csv"
    ))
    tables = []
    for path in sorted(paths):
        parts = path.relative_to(root).parts
        method, version, dataset = parts[:3]
        if dataset == "hest":
            sample_id = parts[3]
            arm, task, probe = parts[5:8]
        else:
            sample_id = pd.NA
            arm, task, probe = parts[4:7]
        table = pd.read_csv(path)
        table["version"] = version
        table["sample_id"] = sample_id
        table["paper_method"] = paper_method(method, version, arm)
        table["benchmark"] = benchmark_label(dataset, task)
        table["result_dir"] = str(path.parent)
        tables.append(table)
    if not tables:
        raise FileNotFoundError("No conformal metrics found below {}".format(root))
    table = pd.concat(tables, ignore_index=True)
    table = table.loc[table["method"] != "hibou_l"]
    table = table.loc[~((table["method"] == "lemon") & (table["version"] != "vits8"))]
    table = table.loc[~table["sample_id"].astype(str).isin(EXCLUDED_HEST_SAMPLES)]
    table["normalized_set_size"] = (
        table["mean_prediction_set_size"] / table["n_classes"]
    )
    return table.reset_index(drop=True)


def aggregate_points(table):
    return (
        table.groupby(
            ["paper_method", "benchmark", "probe", "conformal_method"],
            as_index=False,
        )
        .agg(
            marginal_coverage=("marginal_coverage", "mean"),
            normalized_set_size=("normalized_set_size", "mean"),
            mean_set_size=("mean_prediction_set_size", "mean"),
            class_coverage_gap=("coverage_gap", "mean"),
            n_hest_sections=("sample_id", "count"),
        )
    )


conformal = load_conformal_metrics(RESULTS_ROOT)
points = aggregate_points(conformal)
print("Loaded {:,} conformal rows; probe = {}; excluded HEST = {}".format(
    len(conformal), PROBE, sorted(EXCLUDED_HEST_SAMPLES)
))
display(
    points.loc[
        (points["probe"] == PROBE)
        & (points["paper_method"] == "CellBridge")
        & points["conformal_method"].isin(CP_ORDER)
    ].sort_values(["conformal_method", "benchmark"])
)

## Figure 1 — representation comparison by dataset and in aggregate

Each dataset/task receives its own figure and corresponding displayed table. Color identifies the representation, and the dataset-specific marker is retained for visual consistency. Branch split is fixed. HEST is already the mean across the 14 retained sections. The final figure gives each of the six dataset/tasks equal weight and plots one aggregate point per representation.

In [ ]:
def add_method_legend(ax, marker="o"):
    handles = [
        Line2D(
            [], [], marker=marker, linestyle="", markersize=8,
            markerfacecolor=METHOD_COLORS[name], markeredgecolor="#222222",
            label=name,
        )
        for name in METHOD_ORDER
    ]
    ax.legend(
        handles=handles, title="Representation", frameon=False,
        bbox_to_anchor=(1.02, 1.00), loc="upper left", borderaxespad=0,
    )


def add_two_legends(ax, colors, color_order, color_labels=None):
    color_labels = color_labels or {name: name for name in color_order}
    color_handles = [
        Line2D(
            [], [], marker="o", linestyle="", markersize=8,
            markerfacecolor=colors[name], markeredgecolor="#222222",
            label=color_labels[name],
        )
        for name in color_order
    ]
    dataset_handles = [
        Line2D(
            [], [], marker=DATASET_MARKERS[name], linestyle="", markersize=8,
            markerfacecolor="white", markeredgecolor="#222222", label=name,
        )
        for name in DATASET_ORDER
    ]
    first = ax.legend(
        handles=color_handles, title="Representation", frameon=False,
        bbox_to_anchor=(1.02, 1.00), loc="upper left", borderaxespad=0,
    )
    ax.add_artist(first)
    ax.legend(
        handles=dataset_handles, title="Dataset / task", frameon=False,
        bbox_to_anchor=(1.02, 0.48), loc="upper left", borderaxespad=0,
    )


def padded_limits(values, fraction=0.06, minimum_width=0.02, bounds=None):
    values = np.asarray(values, dtype=float)
    width = max(float(np.ptp(values)), minimum_width)
    limits = (values.min() - fraction * width, values.max() + fraction * width)
    if bounds is not None:
        limits = (max(bounds[0], limits[0]), min(bounds[1], limits[1]))
    return limits


def style_validity_axis(ax, title, x_limits=None, y_limits=None):
    ax.axhline(90, color="#555555", linestyle="--", linewidth=1.2, zorder=1)
    ax.text(
        0.985, 90.25, r"90\% target", color="#555555", ha="right",
        transform=ax.get_yaxis_transform(),
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 1},
    )
    ax.set(
        xlabel="Mean prediction-set size / number of eligible labels",
        ylabel=r"Marginal coverage (\%)",
        title=title,
    )
    if x_limits is not None:
        ax.set_xlim(x_limits)
    if y_limits is not None:
        ax.set_ylim(y_limits)
    lower, upper = ax.get_ylim()
    if upper < 90.6:
        ax.set_ylim(lower, 90.6)
    ax.grid(alpha=0.16, linewidth=0.7)


def plot_representation_rows(rows, title, filename, marker, x_limits=None, y_limits=None):
    fig, ax = plt.subplots(figsize=(8.8, 5.8))
    rows = rows.set_index("paper_method")
    for method in METHOD_ORDER:
        if method not in rows.index:
            continue
        row = rows.loc[method]
        ax.scatter(
            row["normalized_set_size"], 100 * row["marginal_coverage"],
            s=110, marker=marker, facecolor=METHOD_COLORS[method],
            edgecolor="#222222", linewidth=0.7, alpha=0.95, zorder=3,
        )
    style_validity_axis(ax, title, x_limits, y_limits)
    add_method_legend(ax, marker)
    fig.subplots_adjust(right=0.70)
    if SAVE_FIGURES:
        for suffix in ("png", "pdf"):
            fig.savefig(
                FIGURE_DIR / "{}.{}".format(filename, suffix),
                bbox_inches="tight",
            )
    plt.show()


method_rows = points.loc[
    (points["probe"] == PROBE)
    & (points["conformal_method"] == METHOD_COMPARISON_CP)
    & points["paper_method"].isin(METHOD_ORDER)
].copy()

counts = method_rows.groupby("paper_method")["benchmark"].nunique()
if not (counts.reindex(METHOD_ORDER) == len(DATASET_ORDER)).all():
    raise ValueError("Every plotted representation must have all six dataset/tasks")

dataset_slugs = {
    "HEST": "hest",
    "Lizard": "lizard",
    "NuCLS": "nucls",
    "PanNuke": "pannuke",
    "PhenoCell (all)": "phenocell_all",
    "PhenoCell (immune)": "phenocell_immune",
}

for benchmark in DATASET_ORDER:
    rows = method_rows.loc[method_rows["benchmark"] == benchmark]
    plot_representation_rows(
        rows,
        "{} representation comparison under branch split ({})".format(benchmark, PROBE),
        "representation_comparison_{}_{}".format(dataset_slugs[benchmark], PROBE),
        DATASET_MARKERS[benchmark],
        padded_limits(rows["normalized_set_size"], fraction=0.10, bounds=(0, 1)),
        padded_limits(
            np.append(100 * rows["marginal_coverage"].to_numpy(), 90),
            fraction=0.10,
            minimum_width=0.5,
            bounds=(0, 100),
        ),
    )

mean_rows = (
    method_rows.groupby("paper_method", as_index=False)
    .agg(
        marginal_coverage=("marginal_coverage", "mean"),
        normalized_set_size=("normalized_set_size", "mean"),
    )
)
plot_representation_rows(
    mean_rows,
    "Representation comparison: equal-weight mean across datasets ({})".format(PROBE),
    "representation_comparison_mean_{}".format(PROBE),
    "*",
    padded_limits(mean_rows["normalized_set_size"], bounds=(0, 1)),
    padded_limits(
        np.append(100 * mean_rows["marginal_coverage"].to_numpy(), 90),
        minimum_width=1,
        bounds=(0, 100),
    ),
)

display(
    mean_rows.set_index("paper_method")
    .reindex(METHOD_ORDER)
    .assign(coverage_pct=lambda x: 100 * x["marginal_coverage"])
    [["coverage_pct", "normalized_set_size"]]
)


## Figure 2 — conformal construction comparison

CellBridge is fixed, with its per-dataset and aggregate values displayed as a table. Colors identify class split, global graph, and branch split; dataset shapes retain the same meaning used throughout the notebook. The intended reading is the leftmost point close to or above the 90% target, rather than the point with the greatest overcoverage.

In [ ]:
cp_rows = points.loc[
    (points["probe"] == PROBE)
    & (points["paper_method"] == "CellBridge")
    & points["conformal_method"].isin(CP_ORDER)
].copy()

fig, ax = plt.subplots(figsize=(10.7, 6.2))
for _, row in cp_rows.iterrows():
    ax.scatter(
        row["normalized_set_size"], 100 * row["marginal_coverage"],
        s=105, marker=DATASET_MARKERS[row["benchmark"]],
        facecolor=CP_COLORS[row["conformal_method"]],
        edgecolor="#222222", linewidth=0.7, alpha=0.95, zorder=3,
    )
ax.axhline(90, color="#555555", linestyle="--", linewidth=1.2, zorder=1)
ax.text(0.99, 90.25, r"90\% target", color="#555555", ha="right",
        transform=ax.get_yaxis_transform(),
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 1})
ax.set(
    xlabel="Mean prediction-set size / number of eligible labels",
    ylabel=r"Marginal coverage (\%)",
    title="Conformal constructions for CellBridge ({})".format(PROBE),
)
ax.grid(alpha=0.16, linewidth=0.7)
add_two_legends(ax, CP_COLORS, CP_ORDER, CP_LABELS)
ax.get_legend().set_title("Dataset / task")
# Rename the first legend because colors represent conformal constructions here.
for artist in ax.artists:
    if isinstance(artist, plt.matplotlib.legend.Legend):
        artist.set_title("Conformal construction")
fig.subplots_adjust(right=0.72)
if SAVE_FIGURES:
    for suffix in ("png", "pdf"):
        fig.savefig(
            FIGURE_DIR / "conformal_type_comparison_{}.{}".format(PROBE, suffix),
            bbox_inches="tight",
        )
plt.show()

display(
    cp_rows.groupby("conformal_method")
    .agg(
        coverage_pct=("marginal_coverage", lambda x: 100 * x.mean()),
        normalized_set_size=("normalized_set_size", "mean"),
        class_coverage_gap_pp=("class_coverage_gap", "mean"),
    )
    .reindex(CP_ORDER)
    .rename(index=CP_LABELS)
)

## Figure 3 — spatial uncertainty by prediction-set cardinality

The H&E background and points below show a dense square crop within the held-out spatial test region. The crop is selected using cell density only, independently of uncertainty. Cardinality is the number of cell labels retained for an individual cell: one is decisive, while larger sets indicate increasing ambiguity. A sequential colormap is used because cardinality is ordered; the categorical paper palette is reserved for methods and conformal constructions.

In [ ]:
SET_KEYS = {
    "split": "split_sets",
    "graph": "graph_sets",
    "branch_split": "branch_split_sets",
}

spatial_row = conformal.loc[
    (conformal["paper_method"] == "CellBridge")
    & (conformal["dataset"] == "hest")
    & (conformal["sample_id"].astype(str) == SPATIAL_SAMPLE)
    & (conformal["probe"] == PROBE)
    & (conformal["conformal_method"] == SPATIAL_CP)
].iloc[0]
prediction_path = Path(spatial_row["result_dir"]) / "predictions.npz"
if not prediction_path.exists():
    prediction_path = (
        Path(spatial_row["result_dir"]).parent
        / SPATIAL_SAMPLE / "conformal_prediction/predictions.npz"
    )
with np.load(prediction_path) as prediction:
    cell_ids = prediction["ids"].astype(str)
    set_sizes = prediction[SET_KEYS[SPATIAL_CP]].sum(axis=1).astype(int)

adata_path = DATA_ROOT / "train/hest" / SPATIAL_SAMPLE / "{}.h5ad".format(SPATIAL_SAMPLE)
adata = ad.read_h5ad(adata_path, backed="r")
coordinates = adata.obs[["cell_id", "he_x", "he_y"]].copy()
coordinates["cell_id"] = coordinates["cell_id"].astype(str)
coordinates = coordinates.set_index("cell_id").reindex(cell_ids)
adata.file.close()
if coordinates[["he_x", "he_y"]].isna().any().any():
    raise ValueError("Some conformal prediction IDs lack H&E coordinates")
xy = coordinates[["he_x", "he_y"]].to_numpy(float)

manifest = pd.read_csv(DATA_ROOT / "train/hest/manifest.csv", dtype={"id": str})
wsi_path = Path(manifest.loc[manifest["id"] == SPATIAL_SAMPLE, "wsi"].iloc[0])
with tifffile.TiffFile(wsi_path) as slide:
    levels = slide.series[0].levels
    level = min(HISTOLOGY_LEVEL, len(levels) - 1)
    base_height, base_width = levels[0].shape[:2]
    thumbnail = levels[level].asarray()

# Choose a fixed-size dense crop without looking at prediction-set cardinality.
def densest_window_center(values, width):
    values = np.sort(np.asarray(values, dtype=float))
    left = 0
    best_left, best_right = 0, 0
    for right in range(len(values)):
        while values[right] - values[left] > width:
            left += 1
        if right - left > best_right - best_left:
            best_left, best_right = left, right
    return 0.5 * (values[best_left] + values[best_right])


def bounded_window(center, width, upper):
    width = min(float(width), float(upper))
    start = float(np.clip(center - width / 2, 0, upper - width))
    return start, start + width


window = min(float(SPATIAL_WINDOW_PX), float(base_width), float(base_height))
x0, x1 = bounded_window(densest_window_center(xy[:, 0], window), window, base_width)
y0, y1 = bounded_window(densest_window_center(xy[:, 1], window), window, base_height)
visible = (
    (xy[:, 0] >= x0) & (xy[:, 0] <= x1)
    & (xy[:, 1] >= y0) & (xy[:, 1] <= y1)
)
plot_xy = xy[visible]
plot_sizes = set_sizes[visible]
scale_x = thumbnail.shape[1] / base_width
scale_y = thumbnail.shape[0] / base_height
ix0, ix1 = int(np.floor(x0 * scale_x)), int(np.ceil(x1 * scale_x))
iy0, iy1 = int(np.floor(y0 * scale_y)), int(np.ceil(y1 * scale_y))
crop = thumbnail[iy0:iy1, ix0:ix1]
extent = (ix0 / scale_x, ix1 / scale_x, iy1 / scale_y, iy0 / scale_y)

minimum, maximum = int(set_sizes.min()), int(set_sizes.max())
boundaries = np.arange(minimum - 0.5, maximum + 1.5)
cmap = plt.get_cmap("viridis", maximum - minimum + 1)
norm = BoundaryNorm(boundaries, cmap.N)
order = np.argsort(plot_sizes)

fig, ax = plt.subplots(figsize=(8.2, 7.2))
ax.imshow(crop, extent=extent, origin="upper")
scatter = ax.scatter(
    plot_xy[order, 0], plot_xy[order, 1], c=plot_sizes[order], cmap=cmap, norm=norm,
    s=4.0, alpha=0.78, linewidths=0, rasterized=True,
)
colorbar = fig.colorbar(scatter, ax=ax, fraction=0.035, pad=0.025)
ticks = np.arange(minimum, maximum + 1)
if len(ticks) > 12:
    ticks = np.unique(np.linspace(minimum, maximum, 8).round().astype(int))
colorbar.set_ticks(ticks)
colorbar.set_label("Prediction-set cardinality")
ax.set_title(
    "CellBridge: {}\n{} dense crop of held-out test region ({})".format(
        CP_LABELS.get(SPATIAL_CP, SPATIAL_CP), SPATIAL_SAMPLE, PROBE
    )
)
ax.set_axis_off()
fig.tight_layout()
if SAVE_FIGURES:
    for suffix in ("png", "pdf"):
        fig.savefig(
            FIGURE_DIR / "spatial_cardinality_{}_{}_{}.{}".format(
                SPATIAL_SAMPLE, SPATIAL_CP, PROBE, suffix
            ),
            bbox_inches="tight",
        )
plt.show()

print("Showing {:,} of {:,} held-out cells.".format(len(plot_sizes), len(set_sizes)))
display(pd.Series(set_sizes).describe().rename("set cardinality (full test region)"))

## Reading the figures

- Coverage is a validity constraint: values much above 90% are conservative, not automatically superior.
- The x-axis is comparable across tasks because cardinality is divided by the number of eligible labels.
- Marker shape carries the dataset identity exactly as requested. If the combined legend proves too dense in the final layout, the same notebook can be changed to facets without changing the aggregation.
- ECDF means empirical cumulative distribution function: a curve giving the fraction of cells whose set cardinality is at most each value. It is not used here; the spatial map is more directly interpretable for the biological story.